In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine('postgresql+psycopg2://lu.kuang:EFTd66Qa3QhnMeA7\
@10.44.113.30/ABeICU')

# Path to the CSV file
csv_file = '/home/lu.kuang/thesis/GithubFolder/jupyterlab/DataCleaningScripts/ProspectiveCleaning/selected_features.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Extract the item prefix before the first "_" or "\" and create a new column
df['Items'] = df['Selected Features'].str.extract(r'^([^_\\]+)')

# Drop duplicate rows (if needed) and get unique item values
items = df['Items'].dropna().drop_duplicates().reset_index(drop=True)

# Display the cleaned list of items
items = pd.DataFrame(items)

In [2]:
items

,Items
0,GENDER
1,I000
2,I002
3,I009
4,I011
5,I012
6,I035
7,I037
8,I039
9,I056


In [3]:
# Connect to the database
items_df = pd.read_sql_query('SELECT * FROM public."ITEMS"',con=engine)
items_df

,ROW_ID,ITEM_ID,LABEL,ABBREVIATION,CATEGORY,PARAM_TYPE,UOM
0,0.0,I000,Eye Opening,Eye Opening,GCS,NUM,None
1,1.0,I001,Verbal Response,Verbal Response,GCS,NUM,None
2,2.0,I002,Glasgow Coma Scale,GCS,GCS,NUM,None
3,3.0,I003,Motor Response,Motor Response,GCS,NUM,None
4,4.0,I004,Verbal Response - Unable to Assess,Verbal Response - Unable to Assess,GCS,CHAR,None
...,...,...,...,...,...,...,...
766,766.0,I766,Comment NIV,Comment NIV,Comments - Mechanical Ventilation,CHAR,None
767,767.0,I767,Comment HFOV,Comment HFOV,Comments - Mechanical Ventilation,CHAR,None
768,768.0,I768,Comment V2,Comment V2,Comments - Mechanical Ventilation,CHAR,None
769,769.0,I769,Out of Range Reason Comment V2,Out of Range Reason Comment V2,Comments - Mechanical Ventilation,CHAR,None


In [4]:
df = items.merge(items_df, left_on='Items', right_on="ITEM_ID")
df

,Items,ROW_ID,ITEM_ID,LABEL,ABBREVIATION,CATEGORY,PARAM_TYPE,UOM
0,I000,0.0,I000,Eye Opening,Eye Opening,GCS,NUM,None
1,I002,2.0,I002,Glasgow Coma Scale,GCS,GCS,NUM,None
2,I009,9.0,I009,Psychomotor Agitation or Retardation,Psychomotor Agitation or Retardation,ICDSC,BIN,None
3,I011,11.0,I011,Altered Level of Consciousness,Altered Level of Consciousness,ICDSC,BIN,None
4,I012,12.0,I012,"Hallucination, Delusion or Psychosis","Hallucination, Delusion or Psychosis",ICDSC,BIN,None
5,I035,35.0,I035,Non-Invasive Blood Pressure - Systolic,NIBP Systolic,Vitals,NUM,mmHg
6,I037,37.0,I037,Non-Invasive Blood Pressure - Mean,NIBP Mean,Vitals,NUM,mmHg
7,I039,39.0,I039,Resp Rate Bedside Monitor,Resp Rate Bedside Monitor,Vitals,NUM,breaths/min
8,I056,56.0,I056,Creatinine Blood,Creatinine Blood,Labs,NUM,umol/L
9,I059,59.0,I059,Sodium Blood,Sodium Blood,Labs,NUM,mmol/L


In [5]:
df.to_csv("items_used_in_training.csv", index=False)